# Data Quality Governance

This notebook provides a consolidated view of data-quality controls already
implemented in the Silver layer.

It does **not** clean, delete, correct, or transform clinical records.

It simply reports:
- which Silver tables contain `dq_*` quality flags
- what quality rules exist
- how many records are flagged by each rule

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType

CATALOG = "cdr_mimiciv"
SCHEMA = "silver"

## 1. Discover Silver Tables

In [0]:
tables = [
    row.tableName
    for row in spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()
]

print(f"Silver tables found: {len(tables)}")

Silver tables found: 39


## 2. Discover Data Quality Rules

Quality columns are identified using the naming convention:

`dq_*`

In [0]:
dq_rules = []

for table_name in tables:

    df = spark.table(f"{CATALOG}.{SCHEMA}.{table_name}")

    for field in df.schema.fields:

        if field.name.startswith("dq_"):

            dq_rules.append({
                "table_name": table_name,
                "rule_name": field.name,
                "data_type": field.dataType.simpleString()
            })

display(spark.createDataFrame(dq_rules))

data_type,rule_name,table_name
boolean,dq_invalid_admission_time,admissions
boolean,dq_invalid_ed_time,admissions
boolean,dq_death_before_admission,admissions
boolean,dq_death_after_discharge,admissions
boolean,dq_expire_deathtime_mismatch,admissions
boolean,dq_missing_observation,chartevents
boolean,dq_storetime_before_charttime,chartevents
boolean,dq_storetime_before_charttime,datetimeevents
boolean,dq_result_not_reported,labevents
boolean,dq_storetime_before_charttime,labevents


## 3. Calculate Quality Results

In [0]:
results = []

for table_name in tables:

    full_table = f"{CATALOG}.{SCHEMA}.{table_name}"
    df = spark.table(full_table)

    dq_columns = [
        field
        for field in df.schema.fields
        if field.name.startswith("dq_")
    ]

    if not dq_columns:
        continue

    total_rows = df.count()

    for field in dq_columns:

        rule = field.name

        # Most project DQ flags are expected to be Boolean.
        if isinstance(field.dataType, BooleanType):

            failed_rows = (
                df
                .filter(F.col(rule) == True)
                .count()
            )

        else:
            # Allows simple 1/0 DQ flags if present.
            failed_rows = (
                df
                .filter(F.col(rule).cast("int") == 1)
                .count()
            )

        failure_pct = (
            (failed_rows / total_rows) * 100
            if total_rows > 0
            else 0
        )

        results.append({
            "table_name": table_name,
            "rule_name": rule,
            "total_rows": total_rows,
            "failed_rows": failed_rows,
            "failure_pct": round(failure_pct, 4)
        })

## 4. Consolidated Data Quality Summary

In [0]:
if results:

    quality_summary = (
        spark.createDataFrame(results)
        .orderBy(
            F.desc("failed_rows"),
            "table_name",
            "rule_name"
        )
    )

    display(quality_summary)

else:

    print("No dq_* quality flags were found in the Silver layer.")

failed_rows,failure_pct,rule_name,table_name,total_rows
36530969,8.4368,dq_storetime_before_charttime,chartevents,432997491
757757,14.1389,dq_storetime_before_charttime,outputevents,5359395
459636,4.6057,dq_storetime_before_charttime,datetimeevents,9979761
276992,0.5305,dq_ordertime_before_previous_seq,poe,52212109
41504,1.0407,dq_result_not_reported,microbiologyevents,3988224
38501,0.0737,dq_discontinue_of_poe_id_not_in_poe,poe,52212109
1667,0.0032,dq_discontinued_by_poe_id_not_in_poe,poe,52212109
1665,0.0011,dq_result_not_reported,labevents,158374764
1372,9.0E-4,dq_storetime_before_charttime,labevents,158374764
1092,0.0141,dq_result_value_unparsed,omr,7753027


## 5. Quality Coverage

Tables without explicit `dq_*` flags are also shown so that future teams can
identify where additional quality controls may be required.

In [0]:
coverage = []

for table_name in tables:

    df = spark.table(f"{CATALOG}.{SCHEMA}.{table_name}")

    dq_columns = [
        field.name
        for field in df.schema.fields
        if field.name.startswith("dq_")
    ]

    coverage.append({
        "table_name": table_name,
        "dq_rule_count": len(dq_columns),
        "dq_rules": ", ".join(dq_columns) if dq_columns else "None"
    })

coverage_df = (
    spark.createDataFrame(coverage)
    .orderBy("table_name")
)

display(coverage_df)

dq_rule_count,dq_rules,table_name
5,"dq_invalid_admission_time, dq_invalid_ed_time, dq_death_before_admission, dq_death_after_discharge, dq_expire_deathtime_mismatch",admissions
0,None,caregiver
2,"dq_missing_observation, dq_storetime_before_charttime",chartevents
0,None,d_hcpcs
0,None,d_icd_diagnoses
0,None,d_icd_procedures
0,None,d_items
0,None,d_labitems
1,dq_storetime_before_charttime,datetimeevents
0,None,diagnoses_icd


In [0]:
print("Data Quality Governance report complete.")

Data Quality Governance report complete.
